In [1]:
#BankAccount class
from datetime import datetime
from decimal import Decimal, InvalidOperation


class BankAccountError(Exception):
    """Base error."""

class InvalidAmountError(BankAccountError):
    """Amount valid nahi."""

class InsufficientFundsError(BankAccountError):
    """Balance kam hai."""


class BankAccount:
    __slots__ = ("__owner", "__balance", "__history")

    def __init__(self, owner, initial_balance=0):
        name = " ".join(str(owner or "").split())
        if not name:
            raise ValueError("Owner name khali nahi ho sakta")
        self.__owner = name.title()
        self.__balance = Decimal("0.00")
        self.__history = []
        opening = self._to_money(initial_balance, allow_zero=True)
        if opening > 0:
            self.__apply("OPENING", opening)

    # ---------- validation ----------
    @staticmethod
    def _to_money(amount, allow_zero=False):
        if isinstance(amount, bool) or not isinstance(amount, (int, float, Decimal, str)):
            raise InvalidAmountError(f"Amount number hona chahiye, mila: {type(amount).__name__}")
        try:
            value = Decimal(str(amount).strip())
        except InvalidOperation:
            raise InvalidAmountError(f"'{amount}' number nahi hai") from None
        if not value.is_finite():
            raise InvalidAmountError("Amount finite number hona chahiye")
        if value < 0 or (value == 0 and not allow_zero):
            raise InvalidAmountError("Amount zero se bara hona chahiye")
        cents = Decimal("0.01")
        if value != value.quantize(cents):
            raise InvalidAmountError("Amount mein 2 se zyada decimal places nahi ho sakte")
        return value.quantize(cents)

    # ---------- private core ----------
    def __apply(self, kind, amount):
        """Balance sirf yahan badalta hai."""
        self.__balance += amount if kind in ("OPENING", "DEPOSIT") else -amount
        self.__history.append((datetime.now(), kind, amount, self.__balance))

    # ---------- public API ----------
    @property
    def owner(self):
        return self.__owner            # read-only: setter nahi hai

    def deposit(self, amount):
        self.__apply("DEPOSIT", self._to_money(amount))

    def withdraw(self, amount):
        value = self._to_money(amount)
        if value > self.__balance:
            raise InsufficientFundsError(
                f"Balance kam hai: withdraw Rs. {value:,.2f}, available Rs. {self.__balance:,.2f}")
        self.__apply("WITHDRAW", value)

    # ---------- display ----------
    def __str__(self):
        line = "=" * 46
        out = [line, "BANK ACCOUNT SUMMARY".center(46), line,
               f"Owner        : {self.__owner}",
               f"Balance      : Rs. {self.__balance:,.2f}",
               f"Transactions : {len(self.__history)}"]
        if self.__history:
            out += ["-" * 46, "Recent activity (last 5):"]
            out += [f"  {t:%H:%M:%S}  {k:<9}{a:>12,.2f}  -> {b:>12,.2f}"
                    for t, k, a, b in self.__history[-5:]]
        out.append(line)
        return "\n".join(out)

    def __repr__(self):
        return f"BankAccount(owner={self.__owner!r})"   # balance repr mein bhi nahi

In [2]:
#Valid aur invalid operations test
acc = BankAccount("malika", 1000)
acc.deposit(500)
acc.withdraw(200)

bad_calls = [
    ("deposit -50",         lambda: acc.deposit(-50)),
    ("deposit 0",           lambda: acc.deposit(0)),
    ("deposit 'abc'",       lambda: acc.deposit("abc")),
    ("deposit True",        lambda: acc.deposit(True)),
    ("deposit nan",         lambda: acc.deposit(float("nan"))),
    ("deposit 10.555",      lambda: acc.deposit(10.555)),
    ("withdraw 5000",       lambda: acc.withdraw(5000)),
    ("withdraw None",       lambda: acc.withdraw(None)),
]

for label, call in bad_calls:
    try:
        call()
        print(f"⚠️ {label}: allow ho gaya (ye nahi hona chahiye)")
    except BankAccountError as e:
        print(f"❌ {label:<18} -> {type(e).__name__}: {e}")

print()
print(acc)

❌ deposit -50        -> InvalidAmountError: Amount zero se bara hona chahiye
❌ deposit 0          -> InvalidAmountError: Amount zero se bara hona chahiye
❌ deposit 'abc'      -> InvalidAmountError: 'abc' number nahi hai
❌ deposit True       -> InvalidAmountError: Amount number hona chahiye, mila: bool
❌ deposit nan        -> InvalidAmountError: Amount finite number hona chahiye
❌ deposit 10.555     -> InvalidAmountError: Amount mein 2 se zyada decimal places nahi ho sakte
❌ withdraw 5000      -> InsufficientFundsError: Balance kam hai: withdraw Rs. 5,000.00, available Rs. 1,300.00
❌ withdraw None      -> InvalidAmountError: Amount number hona chahiye, mila: NoneType

             BANK ACCOUNT SUMMARY             
Owner        : Malika
Balance      : Rs. 1,300.00
Transactions : 3
----------------------------------------------
Recent activity (last 5):
  11:46:56  OPENING      1,000.00  ->     1,000.00
  11:46:56  DEPOSIT        500.00  ->     1,500.00
  11:46:56  WITHDRAW       200.00  

In [3]:
#Balance bahar se chhune ki koshish
def try_access(label, action):
    try:
        action()
        print(f"⚠️ {label}: kaam kar gaya")
    except AttributeError as e:
        print(f"🔒 {label:<26} blocked ({type(e).__name__})")

def set_balance():
    acc.balance = 1_000_000

def set_owner():
    acc.owner = "Hacker"

try_access("acc.balance (parhna)",   lambda: acc.balance)
try_access("acc.__balance (parhna)", lambda: acc.__balance)
try_access("acc.balance = 1000000",  set_balance)
try_access("acc.owner = 'Hacker'",   set_owner)
try_access("naya attribute banana",  lambda: setattr(acc, "bonus", 1))

print("\nRepr:", repr(acc))

🔒 acc.balance (parhna)       blocked (AttributeError)
🔒 acc.__balance (parhna)     blocked (AttributeError)
🔒 acc.balance = 1000000      blocked (AttributeError)
🔒 acc.owner = 'Hacker'       blocked (AttributeError)
🔒 naya attribute banana      blocked (AttributeError)

Repr: BankAccount(owner='Malika')
